In [ ]:
# Script to clone the repository and switch to the desired branch (ONLY FOR COLAB)

!git clone https://github.com/francescopausellii/uniform-coloring-ai.git
%cd uniform-coloring-ai
!git checkout feat/path-finding

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
# INPUT DEL PROBLEMA — due modalità alternative:
#  - INPUT_IMAGE: path di un'immagine da cui riconoscere la griglia (rete neurale)
#  - INPUT_GRID:  griglia definita a mano (usata se INPUT_IMAGE è None)
INPUT_IMAGE = "../grid_imgs/image2.png"
# INPUT_IMAGE = None

INPUT_GRID = [
    ["G", "B", "Y", "G"],
    ["Y", "G", "B", "B"],
    ["B", "Y", "G", "Y"],
    ["T", "B", "Y", "G"],
]

In [ ]:
from grid_recognition import show_grid_matrix

if INPUT_IMAGE is not None:
    # Riconoscimento della griglia dall'immagine
    import keras
    from grid_recognition import (
        preparation,
        extract_grid_mask,
        detect_segments,
        classify_segments,
        segment_representative_position,
        cluster_lines_by_position,
        filter_clusters_by_span,
        build_grid_lines,
        compute_intersections,
        extract_all_cells,
        predict_grid,
        show_preprocessing,
        show_line_detection,
        show_recognition_result,
    )

    # Modello di riconoscimento lettere addestrato in model.ipynb
    model = keras.models.load_model("../models/dense.keras")

    # Carica l'immagine, la converte in scala di grigi e ne estrae i contorni (Canny)
    img, gray, edges = preparation(INPUT_IMAGE)

    # Isola le linee della griglia eliminando i tratti delle lettere
    grid_mask = extract_grid_mask(gray)
    show_preprocessing(img, gray, edges, grid_mask)

    # Rileva segmenti di linee con Hough, classificandoli in orizzontali e verticali
    segs = detect_segments(grid_mask)
    horiz, vert = classify_segments(segs)

    # Raggruppa i segmenti per posizione, scarta i tratti di lettere e fitta le linee
    horiz_cl = cluster_lines_by_position(segment_representative_position(horiz, "y"))
    vert_cl = cluster_lines_by_position(segment_representative_position(vert, "x"))
    horiz_cl, vert_cl = filter_clusters_by_span(horiz_cl, vert_cl)
    h_lines, v_lines = build_grid_lines(horiz_cl, vert_cl, img.shape)

    # Calcola le intersezioni tra le linee della griglia per ottenere i vertici delle celle
    pts = compute_intersections(h_lines, v_lines)
    show_line_detection(img, segs, horiz, vert, h_lines, v_lines, pts)

    # Ritaglia, raddrizza in prospettiva ed elimina i bordi neri da ogni quadratino
    cells_2d = extract_all_cells(gray, pts)

    # Passa ogni singola immagine ritagliata alla funzione che si interfaccia con la rete neurale
    grid_matrix = predict_grid(cells_2d, model)

    # Celle estratte con lettera predetta + griglia ricostruita, in un'unica figura
    show_recognition_result(cells_2d, grid_matrix)
else:
    # Griglia fornita a mano, nessun riconoscimento
    grid_matrix = INPUT_GRID
    show_grid_matrix(grid_matrix, "Manually provided grid")

In [ ]:
from uniform_coloring.problem import UniformColoring
from uniform_coloring.search import (
    uniform_cost_search as ucs,
    astar_search as astar,
)

# Problem Initialization: griglia riconosciuta dall'immagine o fornita a mano
problem = UniformColoring(grid_matrix)

In [ ]:
from uniform_coloring.visualization import plot_initial_state, plot_solution

# Initial State Visualization
plot_initial_state(problem)

In [ ]:
# UCS (Uniform Cost Search)
# Optimize the TOTAL COST, considering the weights of colors and movements
sol_ucs = ucs(problem)
plot_solution(problem, sol_ucs, "UCS")

In [ ]:
import time
from uniform_coloring.heuristics import Heuristics

h = Heuristics(problem)
print(f"Target color: {problem.target_color.symbol}")

# A* — color
t0 = time.perf_counter()
sol = astar(problem, h=h.color)
elapsed = time.perf_counter() - t0
plot_solution(problem, sol, f"A* — color ({elapsed:.4f}s)")

In [ ]:
# A* — color_nearest_distance
t0 = time.perf_counter()
sol = astar(problem, h=h.color_nearest_distance)
elapsed = time.perf_counter() - t0
plot_solution(problem, sol, f"A* — color_nearest_distance ({elapsed:.4f}s)")

In [ ]:
# A* — color_nearest_neighbor_distance (not admissible)
t0 = time.perf_counter()
sol = astar(problem, h=h.color_nearest_neighbor_distance)
elapsed = time.perf_counter() - t0
plot_solution(problem, sol, f"A* — color_nearest_neighbor_distance ({elapsed:.4f}s)")

In [ ]:
# A* — minimum_spanning_tree
t0 = time.perf_counter()
sol = astar(problem, h=h.mst)
elapsed = time.perf_counter() - t0
plot_solution(problem, sol, f"A* — MST ({elapsed:.4f}s)")

In [ ]:
# A* — ideal / TSP exact (slow on large grids)
t0 = time.perf_counter()
sol = astar(problem, h=h.ideal)
elapsed = time.perf_counter() - t0
plot_solution(problem, sol, f"A* — ideal / TSP exact ({elapsed:.4f}s)")

In [ ]:
# A* — heuristic_mismatched_coloring
t0 = time.perf_counter()
sol = astar(problem, h=h.heuristic_mismatched_coloring)
elapsed = time.perf_counter() - t0
plot_solution(problem, sol, f"A* — heuristic_mismatched_coloring ({elapsed:.4f}s)")


In [ ]:
# Selezione interattiva dell'algoritmo + animazione step-by-step della soluzione
import ipywidgets as widgets
from IPython.display import display, HTML
from uniform_coloring.visualization import animate_solution

ALGORITHMS = {
    "UCS": lambda: ucs(problem),
    "A* — color": lambda: astar(problem, h=h.color),
    "A* — color_nearest_distance": lambda: astar(problem, h=h.color_nearest_distance),
    "A* — color_nearest_neighbor_distance": lambda: astar(
        problem, h=h.color_nearest_neighbor_distance
    ),
    "A* — MST": lambda: astar(problem, h=h.mst),
    "A* — ideal (TSP exact)": lambda: astar(problem, h=h.ideal),
    "A* — mismatched_coloring": lambda: astar(
        problem, h=h.heuristic_mismatched_coloring
    ),
}


@widgets.interact(
    algorithm=widgets.Dropdown(options=list(ALGORITHMS), description="Algorithm")
)
def run_and_animate(algorithm):
    sol = ALGORITHMS[algorithm]()
    anim = animate_solution(problem, sol, algorithm)
    if anim is not None:
        display(HTML(anim.to_jshtml()))